# SVM — scaling, C en gamma

**Concept op verzoek van de gebruiker, uitgewerkt met Codex.** Milan moet
de keuzes en eigen bijdrage zelf beoordelen. Alle nieuwe bestanden blijven onder
`solutions/milan`. We gebruiken de cursusmethoden en de oorspronkelijke elf
invoervelden. Geen nieuwe onafhankelijke eindtest of menselijke review geclaimd.
De dure training is afzonderlijk werkelijk uitgevoerd; onderstaande rapportcellen
    worden van boven naar beneden uitgevoerd; de precieze uitvoeringsmethode staat
in de notebookmetadata en uitvoeringsreceipt.

## Model en zoekruimte
We gebruiken exact dezelfde buitenfolds. Per familie selecteren drie binnenfolds de instellingen. Scaling wordt bij SVM binnen de pipeline geleerd; boommodellen krijgen geen onnodige standaardisatie.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'results_v2/summary.csv').exists())
sys.path.insert(0, str(HERE))
from mushroom_alternative import development_data, scores
from improve import experiments, make_pipeline, Measurements
X, y, ids, folds = development_data()
summary = pd.read_csv(HERE/'results_v2/summary.csv')
details = pd.read_csv(HERE/'results_v2/outer_fold_metrics.csv')
selections = json.loads((HERE/'results_v2/selections.json').read_text())
families = ['svm']
for family in families:
    estimator, grid = experiments()[family]
    display(estimator)
    print('Zoekruimte:', grid)


Zoekruimte: {'classifier__C': [1, 10, 100], 'classifier__gamma': ['scale', 0.01]}


Pipeline(steps=[('measurements', Measurements()),
                ('preprocessor',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['cap_diameter',
                                                   'stem_height',
                                                   'stem_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['spore_print_color',
                                                   'gill_color', 'habitat',
                                                   'season', 'ring_type',
                                                   'cap_shape', 'stem_surface',
                                                   'jumbled_noise_1'])])),
                ('classifier',
                 SVC(class_weight='balanced', probability=True,
                     random_state=42))])

## De werkelijk uitgevoerde selectie
Deze tabel geeft per buitenfold de binnen training gekozen instellingen en drempel. Een binnenscore is een selectiescore, niet de buitenmeting.

In [2]:
display(pd.DataFrame([s for s in selections if s['family'] in families]))
for family in families:
    display(details[details.model.str.startswith(family+'_')])


,family,fold,best_parameters,inner_cv_macro_f1,f1_threshold,seconds
0,svm,1,"{'classifier__C': 10, 'classifier__gamma': 'sc...",0.688875,0.40,25.738950
1,svm,2,"{'classifier__C': 10, 'classifier__gamma': 'sc...",0.689551,0.39,26.275163
2,svm,3,"{'classifier__C': 10, 'classifier__gamma': 'sc...",0.694314,0.40,26.902956
3,svm,4,"{'classifier__C': 1, 'classifier__gamma': 'sca...",0.679070,0.42,22.583354
4,svm,5,"{'classifier__C': 100, 'classifier__gamma': 0.01}",0.689458,0.40,17.175549


,model,fold,threshold,accuracy,macro_f1,balanced_accuracy,recall_p,precision_p,specificity_e,roc_auc,average_precision,tn,fp,fn,tp
35,svm_default,1,0.50,0.721951,0.702435,0.700984,0.614148,0.638796,0.787819,0.737898,0.667827,401,108,120,191
36,svm_f1_threshold,1,0.40,0.706098,0.689237,0.690090,0.623794,0.610063,0.756385,0.737898,0.667827,385,124,117,194
37,svm_default,2,0.50,0.714634,0.695010,0.693839,0.607717,0.627907,0.779961,0.755987,0.666656,397,112,122,189
38,svm_f1_threshold,2,0.39,0.702439,0.689075,0.692771,0.652733,0.598820,0.732809,0.755987,0.666656,373,136,108,203
39,svm_default,3,0.50,0.730488,0.712144,0.711227,0.632258,0.646865,0.790196,0.770750,0.704086,403,107,114,196
40,svm_f1_threshold,3,0.40,0.734146,0.719726,0.721758,0.670968,0.641975,0.772549,0.770750,0.704086,394,116,102,208
41,svm_default,4,0.50,0.743902,0.729373,0.730867,0.677419,0.656250,0.784314,0.789889,0.717543,400,110,100,210
42,svm_f1_threshold,4,0.42,0.742683,0.727592,0.728621,0.670968,0.656151,0.786275,0.789889,0.717543,401,109,102,208
43,svm_default,5,0.50,0.695122,0.679688,0.682163,0.629032,0.590909,0.735294,0.749557,0.668907,375,135,115,195
44,svm_f1_threshold,5,0.40,0.692683,0.678539,0.682100,0.638710,0.585799,0.725490,0.749557,0.668907,370,140,112,198


## Resultaten tegenover RF300
Standaardlabels en een afgestemde macro-F1-beslisregel staan apart. Die drempel is niet op buitenvalidatie gekozen. Grotere recall heeft alleen betekenis samen met precision, FN en FP.

In [3]:
display(summary[summary.model.eq('rf300_reference_default') | summary.model.str.startswith(tuple(families))][['model','macro_f1_mean','macro_f1_sd','recall_p_mean','precision_p_mean','fn_total','fp_total']])


,model,macro_f1_mean,macro_f1_sd,recall_p_mean,precision_p_mean,fn_total,fp_total
0,rf300_reference_default,0.761876,0.021452,0.604390,0.786176,614,256
7,svm_default,0.703730,0.018602,0.632115,0.632145,571,572
8,svm_f1_threshold,0.700834,0.021463,0.651434,0.618562,541,625


## Opnieuw trainen
De opgeslagen training herhalen kan met onderstaande vlag. Standaard worden alleen de echte resultaten gelezen, zonder onnodig opnieuw te fitten.

In [4]:
RERUN = False
if RERUN:
    import joblib
    from improve import run
    with joblib.parallel_config(backend='threading'):
        run(families)
else:
    print('Rapportcontrole; geen nieuwe training gestart.')


Rapportcontrole; geen nieuwe training gestart.
